## Example 1: Formula parsing and design matrices

In [ ]:
import patsy
import numpy as np
import pandas as pd

# Example 1: Basic Formula Parsing and Design Matrix Creation
try:
    # Create sample data
    data = {
        'y': [1, 3, 2, 5, 4, 6, 3, 7],
        'x1': [1, 2, 3, 4, 5, 6, 7, 8],
        'x2': [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0],
        'category': ['A', 'B', 'A', 'B', 'A', 'B', 'A', 'B']
    }
    df = pd.DataFrame(data)
    
    print("Sample data:")
    print(df)
    
    # Basic formula parsing
    print(f"\n--- Basic Formula Parsing ---")
    
    # Simple linear formula
    formula = "y ~ x1 + x2"
    y, X = patsy.dmatrices(formula, df)
    
    print(f"Formula: {formula}")
    print(f"Response variable shape: {y.shape}")
    print(f"Design matrix shape: {X.shape}")
    print(f"Design matrix columns: {X.design_info.column_names}")
    
    # Show the actual matrices
    print(f"\nResponse variable (y):\n{y[:5]}")  # First 5 values
    print(f"\nDesign matrix (X):\n{X[:5]}")      # First 5 rows
    
    # Parse formula components
    # API drift: `patsy.parse_formula` is a MODULE, not a function — calling it raises
    # `TypeError: 'module' object is not callable`. The public parse entry point is
    # `ModelDesc.from_formula`, which returns the parsed term lists.
    parsed = patsy.ModelDesc.from_formula(formula)
    print(f"\nParsed formula:")
    print(f"  Left side: {parsed.lhs_termlist}")
    print(f"  Right side: {parsed.rhs_termlist}")
    
    # Different formula styles
    formulas = [
        "y ~ x1",                    # Simple regression
        "y ~ x1 + x2",              # Multiple regression  
        "y ~ x1 * x2",              # Interaction
        "y ~ x1 + I(x1**2)",        # Polynomial
        "y ~ category"               # Categorical
    ]
    
    print(f"\n--- Formula Variations ---")
    for f in formulas:
        try:
            _, X_temp = patsy.dmatrices(f, df)
            print(f"'{f}' -> {X_temp.shape} columns: {X_temp.design_info.column_names}")
        except Exception as formula_error:
            print(f"'{f}' -> Error: {formula_error}")
            raise
    
    print("Example 1: Basic formula parsing and design matrices completed")
    
except Exception as e:
    print(f"Error in basic formula parsing: {e}")
    raise AssertionError(
        f"patsy example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Categorical variables and contrasts

In [ ]:
import patsy
import pandas as pd
import numpy as np

# Example 2: Categorical Variables and Contrasts
try:
    # Create data with multiple categorical variables
    np.random.seed(42)
    n = 20
    
    data = pd.DataFrame({
        'response': np.random.normal(10, 2, n),
        'treatment': np.random.choice(['Control', 'DrugA', 'DrugB'], n),
        'gender': np.random.choice(['M', 'F'], n),
        'age_group': np.random.choice(['Young', 'Middle', 'Old'], n),
        'dose': np.random.uniform(1, 10, n)
    })
    
    print("Categorical data sample:")
    print(data.head())
    print(f"\nData shape: {data.shape}")
    
    # Examine categorical variable levels
    print(f"\n--- Categorical Variable Levels ---")
    for col in ['treatment', 'gender', 'age_group']:
        levels = data[col].unique()
        print(f"{col}: {sorted(levels)}")
    
    # Basic categorical encoding
    print(f"\n--- Basic Categorical Encoding ---")
    
    formula = "response ~ treatment + gender"
    y, X = patsy.dmatrices(formula, data)
    
    print(f"Formula: {formula}")
    print(f"Design matrix shape: {X.shape}")
    print(f"Columns: {X.design_info.column_names}")
    print(f"Design matrix sample:\n{X[:5]}")
    
    # Different contrast types
    print(f"\n--- Contrast Types ---")
    
    contrast_formulas = [
        "response ~ C(treatment)",                    # Default contrasts
        "response ~ C(treatment, Treatment)",        # Treatment contrasts (explicit)
        "response ~ C(treatment, Sum)",              # Sum contrasts
    ]
    
    for formula in contrast_formulas:
        try:
            _, X_contrast = patsy.dmatrices(formula, data)
            print(f"\n'{formula}':")
            print(f"  Columns: {X_contrast.design_info.column_names}")
            print(f"  Shape: {X_contrast.shape}")
        except Exception as contrast_error:
            print(f"  Error: {contrast_error}")
            raise
    
    # Multi-level categorical interactions
    print(f"\n--- Categorical Interactions ---")
    
    interaction_formula = "response ~ treatment * gender"
    _, X_interact = patsy.dmatrices(interaction_formula, data)
    
    print(f"Interaction formula: {interaction_formula}")
    print(f"Interaction matrix shape: {X_interact.shape}")
    print(f"Interaction columns: {X_interact.design_info.column_names}")
    
    # Custom contrasts
    print(f"\n--- Custom Contrasts ---")
    
    try:
        # Define custom contrast matrix for treatment
        treatment_levels = sorted(data['treatment'].unique())
        print(f"Treatment levels: {treatment_levels}")
        
        # Create a contrast that compares drugs to control
        if len(treatment_levels) == 3:  # Control, DrugA, DrugB
            custom_contrast = np.array([
                [-0.5, -0.5, 1.0],  # Control vs (DrugA + DrugB)/2
                [1.0, -1.0, 0.0]    # DrugA vs DrugB
            ]).T
            
            print(f"Custom contrast matrix shape: {custom_contrast.shape}")
            print(f"Custom contrast:\n{custom_contrast}")
        
    except Exception as custom_error:
        print(f"Custom contrast error: {custom_error}")
        raise
    
    print("Example 2: Categorical variables and contrasts completed")
    
except Exception as e:
    print(f"Error in categorical variables: {e}")
    raise AssertionError(
        f"patsy example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: Transformations and functions in formulas

In [ ]:
import patsy
import numpy as np
import pandas as pd

# Example 3: Advanced Transformations and Functions in Formulas
try:
    # Generate sample data with various relationships
    np.random.seed(123)
    n = 30
    
    data = pd.DataFrame({
        'y': np.random.normal(0, 1, n),
        'x': np.random.uniform(1, 10, n),
        'z': np.random.normal(5, 2, n),
        'w': np.random.exponential(2, n),
        'group': np.random.choice(['A', 'B', 'C'], n)
    })
    
    print("Data for transformations:")
    print(data.head())
    print(f"Data summary:")
    print(data.describe())
    
    # Built-in transformation functions
    print(f"\n--- Built-in Transformations ---")
    
    transformation_formulas = [
        "y ~ np.log(x)",                    # Logarithm
        "y ~ np.sqrt(x)",                   # Square root
        "y ~ I(x**2)",                      # Square (Identity function)
        "y ~ I(x**2) + I(x**3)",           # Polynomial
        "y ~ np.log(x) + np.sqrt(z)",      # Multiple transformations
        "y ~ center(x)",                    # Centering
        "y ~ standardize(x)",               # Standardization
    ]
    
    for formula in transformation_formulas:
        try:
            _, X_trans = patsy.dmatrices(formula, data)
            print(f"'{formula}':")
            print(f"  Shape: {X_trans.shape}")
            print(f"  Columns: {X_trans.design_info.column_names}")
            
            # Show sample values for first non-intercept column
            if X_trans.shape[1] > 1:
                values = X_trans[:5, 1]  # First 5 values of second column
                print(f"  Sample values: {values.flatten()}")
            print()
            
        except Exception as trans_error:
            print(f"'{formula}' -> Error: {trans_error}\n")
            raise
    
    # Custom transformation functions
    print(f"\n--- Custom Transformations ---")
    
    def custom_transform(x):
        """Custom transformation: sigmoid-like function"""
        return 1 / (1 + np.exp(-x))
    
    def winsorize(x, limits=(0.05, 0.95)):
        """Custom winsorization function"""
        lower, upper = np.quantile(x, limits)
        return np.clip(x, lower, upper)
    
    # Register custom functions in namespace
    custom_namespace = {
        'sigmoid': custom_transform,
        'winsorize': winsorize
    }
    
    custom_formulas = [
        "y ~ sigmoid(x)",
        "y ~ winsorize(w)",
        "y ~ sigmoid(x) + winsorize(w)"
    ]
    
    for formula in custom_formulas:
        try:
            # `eval_env` takes an int (stack depth) or an EvalEnvironment — a plain dict is refused.
            _, X_custom = patsy.dmatrices(formula, data, eval_env=patsy.EvalEnvironment([custom_namespace]))
            print(f"'{formula}':")
            print(f"  Shape: {X_custom.shape}")
            print(f"  Columns: {X_custom.design_info.column_names}")
            expected_columns = ["Intercept"] + [term.strip() for term in formula.split("~")[1].split("+")]
            assert X_custom.design_info.column_names == expected_columns, X_custom.design_info.column_names
            if "sigmoid(x)" in expected_columns:
                sigmoid_column = expected_columns.index("sigmoid(x)")
                assert np.allclose(np.asarray(X_custom)[:, sigmoid_column], custom_transform(data['x'].values))
            
            if X_custom.shape[1] > 1:
                values = X_custom[:3, 1]  # First 3 values
                print(f"  Sample values: {values.flatten()}")
            print()
            
        except Exception as custom_trans_error:
            print(f"'{formula}' -> Error: {custom_trans_error}\n")
            raise
    
    # Interaction with transformations
    print(f"\n--- Interactions with Transformations ---")
    
    interaction_trans_formulas = [
        "y ~ x * np.log(z)",                # Interaction with transformation
        "y ~ group * I(x**2)",              # Categorical-continuous interaction
        "y ~ C(group) * center(x)",         # Categorical with centered continuous
    ]
    
    for formula in interaction_trans_formulas:
        try:
            _, X_int_trans = patsy.dmatrices(formula, data)
            print(f"'{formula}':")
            print(f"  Shape: {X_int_trans.shape}")
            print(f"  Columns: {X_int_trans.design_info.column_names[:5]}...")  # First 5 columns
            print()
            
        except Exception as int_trans_error:
            print(f"'{formula}' -> Error: {int_trans_error}\n")
            raise
    
    # Show transformation effects
    print(f"\n--- Transformation Effects ---")
    
    original_x = data['x'].values[:5]
    log_x = np.log(original_x)
    sqrt_x = np.sqrt(original_x)
    
    print(f"Original x: {original_x}")
    print(f"Log(x):     {log_x}")
    print(f"Sqrt(x):    {sqrt_x}")
    
    print("Example 3: Advanced transformations and functions completed")
    
except Exception as e:
    print(f"Error in transformations: {e}")
    raise AssertionError(
        f"patsy example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: Model building and design matrix inspection

In [ ]:
import patsy
import pandas as pd
import numpy as np

# Example 4: Model Building and Design Matrix Inspection
try:
    # Create realistic example data
    np.random.seed(456)
    n = 50
    
    # Simulate experimental data
    data = pd.DataFrame({
        'outcome': np.random.normal(100, 15, n),
        'treatment': np.random.choice(['Placebo', 'Low', 'Medium', 'High'], n),
        'age': np.random.normal(45, 12, n),
        'gender': np.random.choice(['Male', 'Female'], n),
        'baseline': np.random.normal(80, 20, n),
        'center': np.random.choice(['A', 'B', 'C', 'D'], n)
    })
    
    # Add some realistic relationships
    treatment_effects = {'Placebo': 0, 'Low': 5, 'Medium': 12, 'High': 18}
    for i, row in data.iterrows():
        effect = treatment_effects[row['treatment']]
        data.loc[i, 'outcome'] += effect + 0.3 * row['baseline'] + np.random.normal(0, 5)
    
    print("Experimental data sample:")
    print(data.head())
    print(f"\nTreatment distribution: {data['treatment'].value_counts().to_dict()}")
    
    # Build progressively complex models
    print(f"\n--- Progressive Model Building ---")
    
    models = [
        ("Null model", "outcome ~ 1"),
        ("Treatment only", "outcome ~ treatment"),
        ("Treatment + Age", "outcome ~ treatment + age"),
        ("Treatment + Age + Gender", "outcome ~ treatment + age + gender"),
        ("Full additive", "outcome ~ treatment + age + gender + baseline + center"),
        ("With interactions", "outcome ~ treatment * age + gender + baseline"),
        ("Complex interactions", "outcome ~ treatment * (age + gender) + baseline")
    ]
    
    for model_name, formula in models:
        try:
            y, X = patsy.dmatrices(formula, data)
            
            print(f"\n{model_name}: '{formula}'")
            print(f"  Design matrix shape: {X.shape}")
            print(f"  Parameters: {len(X.design_info.column_names)}")
            print(f"  Columns: {X.design_info.column_names[:4]}..." if len(X.design_info.column_names) > 4 
                  else f"  Columns: {X.design_info.column_names}")
            
            # Show rank (for collinearity detection)
            rank = np.linalg.matrix_rank(X)
            print(f"  Matrix rank: {rank} (full rank: {rank == X.shape[1]})")
            
        except Exception as model_error:
            print(f"\n{model_name} -> Error: {model_error}")
            raise
    
    # Design matrix inspection
    print(f"\n--- Design Matrix Inspection ---")
    
    # Use a moderately complex model for inspection
    inspection_formula = "outcome ~ treatment + age + gender * baseline"
    y_insp, X_insp = patsy.dmatrices(inspection_formula, data)
    
    print(f"Inspection model: '{inspection_formula}'")
    print(f"Design matrix info:")
    print(f"  Shape: {X_insp.shape}")
    print(f"  Data type: {X_insp.dtype}")
    print(f"  Memory usage: {X_insp.nbytes} bytes")
    
    # Column statistics
    print(f"\nColumn statistics:")
    for i, col_name in enumerate(X_insp.design_info.column_names):
        col_data = X_insp[:, i]
        print(f"  {col_name}:")
        print(f"    Mean: {np.mean(col_data):.3f}, Std: {np.std(col_data):.3f}")
        print(f"    Min: {np.min(col_data):.3f}, Max: {np.max(col_data):.3f}")
    
    # Check for perfect collinearity
    print(f"\n--- Collinearity Check ---")
    
    condition_number = np.linalg.cond(X_insp)
    print(f"Condition number: {condition_number:.2f}")
    
    if condition_number > 30:
        print("  Warning: High condition number suggests collinearity")
    else:
        print("  Condition number is acceptable")
    
    # Show term information
    print(f"\n--- Term Information ---")
    
    design_info = X_insp.design_info
    print(f"Terms in model: {len(design_info.terms)}")
    
    for i, term in enumerate(design_info.terms):
        print(f"  Term {i}: {term}")
        columns = design_info.slice(term)
        print(f"    Columns: {columns}")
    
    print("Example 4: Model building and design matrix inspection completed")
    
except Exception as e:
    print(f"Error in model building: {e}")
    raise AssertionError(
        f"patsy example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: Reusing design info to build matrices for new data

In [ ]:
import patsy
import pandas as pd
import numpy as np

# Example 5: Production Workflows and Integration Patterns
def create_model_pipeline(formula_str, data, test_data=None):
    """Production-ready model pipeline using patsy"""
    
    print(f"Building pipeline for formula: '{formula_str}'")
    
    # Parse and validate formula
    try:
        # `patsy.parse_formula` is a MODULE; the public parser is `ModelDesc.from_formula`.
        parsed_formula = patsy.ModelDesc.from_formula(formula_str)
        print(f"✓ Formula parsed successfully")
    except Exception as parse_error:
        print(f"✗ Formula parsing failed: {parse_error}")
        raise
    
    # Build design matrices
    try:
        y_train, X_train = patsy.dmatrices(formula_str, data, return_type='dataframe')
        print(f"✓ Training matrices created: y{y_train.shape}, X{X_train.shape}")
        
        # Store design info for prediction
        design_info = X_train.design_info
        
        # Handle test data if provided
        if test_data is not None:
            X_test = patsy.build_design_matrices([design_info], test_data, return_type='dataframe')[0]
            print(f"✓ Test matrix created: X_test{X_test.shape}")
        else:
            X_test = None
            
        return {
            'y_train': y_train,
            'X_train': X_train,
            'X_test': X_test,
            'design_info': design_info,
            'formula': formula_str
        }
        
    except Exception as build_error:
        print(f"✗ Matrix building failed: {build_error}")
        raise

try:
    print("Production workflows and integration patterns:")
    
    # Generate train/test datasets
    np.random.seed(789)
    
    # Training data
    n_train = 100
    train_data = pd.DataFrame({
        'sales': np.random.gamma(2, 50, n_train),
        'advertising': np.random.uniform(1000, 10000, n_train),
        'price': np.random.uniform(10, 100, n_train),
        'season': np.random.choice(['Spring', 'Summer', 'Fall', 'Winter'], n_train),
        'region': np.random.choice(['North', 'South', 'East', 'West'], n_train),
        'competition': np.random.poisson(3, n_train)
    })
    
    # Test data (new observations)
    n_test = 20
    test_data = pd.DataFrame({
        'advertising': np.random.uniform(1000, 10000, n_test),
        'price': np.random.uniform(10, 100, n_test),
        'season': np.random.choice(['Spring', 'Summer', 'Fall', 'Winter'], n_test),
        'region': np.random.choice(['North', 'South', 'East', 'West'], n_test),
        'competition': np.random.poisson(3, n_test)
    })
    
    print(f"Training data shape: {train_data.shape}")
    print(f"Test data shape: {test_data.shape}")
    
    # Test model pipeline
    print(f"\n--- Model Pipeline Testing ---")
    
    pipeline_formulas = [
        "sales ~ advertising + price",
        "sales ~ advertising * price + season",
        "sales ~ np.log(advertising) + price + C(season) + C(region)",
        "sales ~ I(advertising/1000) + I(price**2) + season * region"
    ]
    
    pipelines = {}
    
    for formula in pipeline_formulas:
        print(f"\n--- Testing Formula: {formula} ---")
        pipeline = create_model_pipeline(formula, train_data, test_data)
        
        if pipeline:
            pipelines[formula] = pipeline
            
            # Show pipeline results
            print(f"Training features: {list(pipeline['X_train'].columns)[:5]}...")
            print(f"Feature consistency: {pipeline['X_train'].shape[1] == pipeline['X_test'].shape[1]}")
            assert pipeline['X_train'].shape[1] == pipeline['X_test'].shape[1], formula
    
    assert len(pipelines) == len(pipeline_formulas), sorted(pipelines)
    
    # Demonstrate prediction workflow
    print(f"\n--- Prediction Workflow ---")
    
    if pipelines:
        # Use first successful pipeline
        example_formula = list(pipelines.keys())[0]
        pipeline = pipelines[example_formula]
        
        print(f"Using pipeline: '{example_formula}'")
        
        # Simulate model fitting (mock regression)
        X_train = pipeline['X_train'].values
        y_train = pipeline['y_train'].values.flatten()
        
        # Simple linear regression for demonstration
        try:
            # Using normal equation: beta = (X'X)^(-1)X'y
            XtX_inv = np.linalg.inv(X_train.T @ X_train)
            beta = XtX_inv @ X_train.T @ y_train
            
            print(f"✓ Model fitted with {len(beta)} parameters")
            # The normal equation must agree with numpy's least-squares solver on the design patsy built.
            assert np.allclose(beta, np.linalg.lstsq(X_train, y_train, rcond=None)[0]), beta
            
            # Make predictions on test set
            X_test = pipeline['X_test'].values
            predictions = X_test @ beta
            
            print(f"✓ Generated {len(predictions)} predictions")
            print(f"Prediction range: [{predictions.min():.1f}, {predictions.max():.1f}]")
            assert len(predictions) == len(test_data) and np.isfinite(predictions).all(), predictions
            
        except np.linalg.LinAlgError as linalg_error:
            # X'X is singular only when the design patsy built is rank-deficient: that is this example failing.
            raise AssertionError(
                f"normal-equation fit failed on the patsy design {list(pipeline['X_train'].columns)}: {linalg_error}"
            ) from linalg_error
    
    # Show formula parsing capabilities
    print(f"\n--- Formula Parsing Capabilities ---")
    
    complex_formula = "y ~ (x1 + x2) * (z1 + z2) + I(w**2) - 1"
    try:
        # `patsy.parse_formula` is a MODULE; the public parser is `ModelDesc.from_formula`.
        parsed = patsy.ModelDesc.from_formula(complex_formula)
        print(f"Complex formula: '{complex_formula}'")
        print(f"Left side terms: {len(parsed.lhs_termlist)}")
        print(f"Right side terms: {len(parsed.rhs_termlist)}")
        rhs_names = [term.name() for term in parsed.rhs_termlist]
        # (x1 + x2) * (z1 + z2) expands to 4 main effects + 4 interactions, plus I(w**2); "- 1" drops the intercept.
        assert len(parsed.lhs_termlist) == 1 and len(rhs_names) == 9, rhs_names
        assert "x2:z1" in rhs_names and "I(w ** 2)" in rhs_names and "Intercept" not in rhs_names, rhs_names
    except Exception as parse_complex_error:
        print(f"Complex parsing error: {parse_complex_error}")
        raise
    
    # Integration with statistical libraries
    print(f"\n--- Statistical Library Integration ---")
    
    integration_notes = [
        "Patsy integrates seamlessly with statsmodels",
        "Design matrices work with scikit-learn",
        "Formula syntax similar to R modeling",
        "Handles categorical variables automatically",
        "Supports custom transformations and functions"
    ]
    
    for i, note in enumerate(integration_notes, 1):
        print(f"{i}. {note}")
    
    # Show patsy version and capabilities
    print(f"\n--- Patsy Information ---")
    
    patsy_attrs = [attr for attr in dir(patsy) if not attr.startswith('_')]
    key_functions = ['dmatrices', 'dmatrix', 'parse_formula', 'build_design_matrices']
    
    print(f"Patsy version: {getattr(patsy, '__version__', 'Unknown')}")
    print(f"Key functions available: {[f for f in key_functions if f in patsy_attrs]}")
    print(f"Total public attributes: {len(patsy_attrs)}")
    
    print("Example 5: Production workflows and integration patterns completed")
    
except Exception as e:
    print(f"Error in production workflows: {e}")
    raise AssertionError(
        f"patsy example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e